
# Bootstrap Regression Confidence Intervals

This notebook is intentionally self-contained. It does not import any local utility module, so you can open this file alone during study and see the full statistical workflow, simulation setup, model fitting, evaluation, plotting, and interpretation pattern in one place.



## What problem are we solving?

Bootstrap confidence intervals estimate uncertainty for regression quantities such as slopes, predictions, or MSE without relying entirely on normal-theory formulas.

**Highest-probability exam pattern:** Bootstrap the regression slope by resampling rows, refitting the model, and taking percentiles of the bootstrapped slopes.



## Assumptions, intuition, and theory

Regression rows are the sampling units, so resample complete (X, y) pairs. A percentile interval uses the empirical quantiles of the bootstrap estimates. Fixed seeds make simulation output reproducible.



    ## Python method notes and documentation

    `LinearRegression.fit` is repeated on bootstrap samples, `coef_` extracts the slope, `np.quantile` creates percentile endpoints, and `statsmodels.OLS` gives a classical reference interval.

    Documentation links:

    - [NumPy random generator](https://numpy.org/doc/stable/reference/random/generator.html)
- [pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [matplotlib pyplot](https://matplotlib.org/stable/api/pyplot_summary.html)
- [scipy.stats](https://docs.scipy.org/doc/scipy/reference/stats.html)
- [OLS](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.html)
- [LinearRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html)
- [PolynomialFeatures](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html)


## Fully self-contained worked example

Every code line below is commented so the workflow can be scanned quickly under exam time pressure.

In [ ]:
import numpy as np  # Import numerical arrays and random-number tools.
import pandas as pd  # Import DataFrame tables for bootstrap summaries.
import matplotlib.pyplot as plt  # Import plotting tools for bootstrap distributions.
import statsmodels.api as sm  # Import statsmodels for OLS reference intervals.
from sklearn.linear_model import LinearRegression  # Import sklearn linear regression for repeated bootstrap fitting.
RANDOM_SEED = 123  # Store the reproducibility seed.
rng = np.random.default_rng(RANDOM_SEED)  # Create a reproducible random generator.
plt.style.use('default')  # Use a predictable plotting style.


In [ ]:
def make_linear_regression_data(n=100, noise=1.2, random_state=123):  # Define a simple linear regression simulator.
    rng = np.random.default_rng(random_state)  # Create a reproducible random generator.
    x = rng.uniform(-3.0, 3.0, size=n)  # Draw one numeric predictor.
    y = 2.0 + 3.0 * x + rng.normal(0.0, noise, size=n)  # Generate response from intercept, slope, and noise.
    X = x.reshape(-1, 1)  # Reshape x for sklearn.
    return X, y  # Return predictors and response.

def bootstrap_slope_ci(X, y, n_boot=2000, confidence=0.95, random_state=123):  # Define a bootstrap CI for the regression slope.
    rng = np.random.default_rng(random_state)  # Create a reproducible random generator.
    X = np.asarray(X)  # Convert predictors to an array.
    y = np.asarray(y)  # Convert response to an array.
    n = len(y)  # Store the sample size.
    slopes = np.empty(n_boot)  # Allocate space for bootstrap slope estimates.
    for b in range(n_boot):  # Repeat bootstrap sampling many times.
        sample_index = rng.integers(0, n, size=n)  # Sample row indices with replacement.
        model = LinearRegression()  # Create a fresh linear regression model.
        model.fit(X[sample_index], y[sample_index])  # Fit the model on the bootstrap sample.
        slopes[b] = model.coef_[0]  # Store the fitted slope.
    alpha = 1.0 - confidence  # Convert confidence level to tail probability.
    lower, upper = np.quantile(slopes, [alpha / 2.0, 1.0 - alpha / 2.0])  # Compute percentile endpoints.
    return lower, upper, slopes  # Return interval endpoints and bootstrap slope distribution.


In [ ]:
X, y = make_linear_regression_data(n=100, noise=1.2, random_state=RANDOM_SEED)  # Simulate a linear regression data set.
base_model = LinearRegression()  # Create the ordinary least-squares model.
base_model.fit(X, y)  # Fit the model to the observed data.
lower, upper, slopes = bootstrap_slope_ci(X, y, n_boot=2000, random_state=RANDOM_SEED)  # Bootstrap the slope confidence interval.
X_with_intercept = sm.add_constant(X)  # Add an intercept column for statsmodels OLS.
ols_result = sm.OLS(y, X_with_intercept).fit()  # Fit the statsmodels OLS reference model.
summary = pd.DataFrame(  # Build a compact interval summary table.
    [  # Start the row list.
        {'method': 'bootstrap percentile', 'slope_estimate': base_model.coef_[0], 'lower': lower, 'upper': upper},  # Add the bootstrap interval row.
        {'method': 'statsmodels t interval', 'slope_estimate': ols_result.params[1], 'lower': ols_result.conf_int()[1, 0], 'upper': ols_result.conf_int()[1, 1]},  # Add the classical OLS interval row.
    ]  # End the row list.
)  # Finish constructing the table.
display(summary)  # Display the interval comparison.
plt.figure(figsize=(6, 4))  # Create a histogram figure for bootstrap slopes.
plt.hist(slopes, bins=30, edgecolor='black', alpha=0.75)  # Plot the bootstrap slope distribution.
plt.axvline(lower, color='red', linestyle='--', label='bootstrap CI lower')  # Mark the lower endpoint.
plt.axvline(upper, color='red', linestyle='--', label='bootstrap CI upper')  # Mark the upper endpoint.
plt.xlabel('bootstrap slope')  # Label the slope axis.
plt.ylabel('count')  # Label the count axis.
plt.title('Bootstrap confidence interval for regression slope')  # Title the plot.
plt.legend()  # Show endpoint labels.
plt.show()  # Render the histogram.



## Exam-style problem prompt

A prompt asks for a confidence interval for a regression slope. Resample rows with replacement, refit the line each time, and report percentile endpoints.



    ## Adaptable code pattern

    ```python
    # Step 1: Fit the original regression model and record the statistic.
# Step 2: Resample rows of the data with replacement.
# Step 3: Refit the regression model on each bootstrap sample.
# Step 4: Store the slope or statistic of interest.
# Step 5: Take percentile endpoints of the bootstrap statistics.
# Step 6: Interpret the interval in the problem context.
    ```



## What to remember for the exam

For regression bootstrapping, resample rows, not individual x and y values separately. Each bootstrap sample should preserve predictor-response pairing.
